<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/08-mini-proyecto-reporte-mensual/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 0.8 · Miniproyecto: automatiza tu reporte mensual

**Serie:** Del Excel al Machine Learning · **Temporada 0** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Unir todo lo aprendido en un solo proyecto, del archivo sucio al reporte final.
- Guardar tu limpieza y tus cálculos en **funciones** que puedas reutilizar.
- Exportar un **libro de Excel** con varias hojas y un gráfico.
- Crear el reporte de cualquier mes cambiando el año y el mes.

## 📊 En Excel lo harías así

Es fin de mes, y te toca el reporte de ventas:

1. Abres la exportación del sistema y la **limpias** a mano.
2. Agregas columnas con `=BUSCARV()` (`VLOOKUP`).
3. Filtras el mes y armas tus **tablas dinámicas**.
4. Copias, pegas y actualizas el **gráfico**.
5. Lo guardas como un libro nuevo.

Un par de horas, y el próximo mes otra vez. Hoy lo escribimos una vez y lo reutilizamos cada mes.

## 🐍 En Python

Casi todo lo que sigue ya lo conoces. Lo nuevo es **cómo lo organizamos**: cada tarea en su función, como las del episodio 0.3.

### Paso 1: preparar y limpiar

Esta vez también traemos `Image`, para meter el gráfico en el Excel.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl.drawing.image import Image

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

Esta función es toda la limpieza de la clase y el ejercicio del 0.5 en un solo lugar. No necesitas leerla línea por línea. Le das un archivo sucio y te devuelve una tabla limpia.

In [2]:
def limpiar_monto(columna):
    return columna.str.replace("S/ ", "").str.replace(",", "").astype(float)


def limpiar_ventas(archivo):
    ventas = pd.read_csv(archivo, sep=";")

    # Nombres de columna
    ventas.columns = ventas.columns.str.lower().str.replace(" ", "_")
    ventas = ventas.rename(
        columns={
            "nro_venta": "id_venta",
            "cliente": "id_cliente",
            "producto": "id_producto",
            "vendedor": "id_vendedor",
            "método_de_pago": "metodo_pago",
        }
    )

    # Duplicados
    ventas = ventas.drop_duplicates()

    # Textos
    ventas["ciudad"] = ventas["ciudad"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].str.strip().str.capitalize()
    ventas["metodo_pago"] = ventas["metodo_pago"].replace(
        {"Yape": "Billetera digital", "Plin": "Billetera digital"}
    )
    ventas["metodo_pago"] = ventas["metodo_pago"].fillna("Sin dato")

    # Números guardados como texto
    ventas["precio_unitario"] = limpiar_monto(ventas["precio_unitario"])
    ventas["total"] = limpiar_monto(ventas["total"])
    ventas["descuento"] = ventas["descuento"].str.replace("%", "").astype(float) / 100

    # Fechas
    texto = ventas["fecha"]
    ventas["fecha"] = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    ventas["fecha"] = ventas["fecha"].fillna(
        pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
    )
    ventas["anio"] = ventas["fecha"].dt.year
    ventas["mes"] = ventas["fecha"].dt.month

    # Cantidades vacías o con errores de tipeo
    cantidad = ventas["total"] / (ventas["precio_unitario"] * (1 - ventas["descuento"]))
    ventas["cantidad"] = cantidad.round().astype(int)

    return ventas

La usamos y traemos los nombres de producto y vendedor con `pd.merge()`. En las dos tablas la columna se llama `nombre`, así que le ponemos un nombre claro con `.rename()`. Y **comprobamos** contra la cifra oficial: 50,000 filas y S/ 13,582,286.61.

In [3]:
ventas = limpiar_ventas(DATOS + "ventas_crudo.csv")

productos = pd.read_csv(DATOS + "productos.csv")
empleados = pd.read_csv(DATOS + "empleados.csv")

productos = productos[["id_producto", "nombre", "categoria"]].rename(columns={"nombre": "producto"})
empleados = empleados[["id_empleado", "nombre"]].rename(columns={"nombre": "vendedor"})

ventas = pd.merge(ventas, productos, on="id_producto", how="left")
ventas = pd.merge(ventas, empleados, left_on="id_vendedor", right_on="id_empleado", how="left")

print(len(ventas))
print(round(ventas["total"].sum(), 2))

50000
13582286.61


### Paso 2: las piezas del reporte

Tres funciones pequeñas:

- `filtrar_mes` se queda con las ventas de un mes.
- `resumir` crea cualquier tabla del reporte: suma por una columna, ordena y calcula qué porcentaje del total representa cada fila. `.reset_index()` convierte los nombres de la izquierda en una columna normal; sin eso, se perderían al exportar.
- `crear_indicadores` calcula los números clave y los compara con el mismo mes del año anterior. `pd.DataFrame()` arma una tabla desde un diccionario: cada clave es una columna, y cada lista, sus valores. El `if` evita dividir entre cero cuando no hay año anterior.

In [4]:
def filtrar_mes(ventas, anio, mes):
    return ventas[(ventas["anio"] == anio) & (ventas["mes"] == mes)]


def resumir(tabla, columna):
    resumen = tabla.groupby(columna)["total"].sum().sort_values(ascending=False).reset_index()
    resumen["porcentaje"] = (resumen["total"] / resumen["total"].sum() * 100).round(1)
    resumen["total"] = resumen["total"].round(2)
    return resumen


def crear_indicadores(ventas, anio, mes):
    del_mes = filtrar_mes(ventas, anio, mes)
    total = del_mes["total"].sum()
    anterior = filtrar_mes(ventas, anio - 1, mes)["total"].sum()

    if anterior > 0:
        variacion = round((total / anterior - 1) * 100, 1)
    else:
        variacion = "Sin dato"

    return pd.DataFrame(
        {
            "indicador": [
                "Período",
                "Ventas totales (S/)",
                "Número de ventas",
                "Venta mediana (S/)",
                "Variación contra el mismo mes del año anterior (%)",
            ],
            "valor": [
                f"{mes:02d}/{anio}",
                round(total, 2),
                len(del_mes),
                del_mes["total"].median(),
                variacion,
            ],
        }
    )

Las probamos con diciembre de 2025:

In [5]:
print(crear_indicadores(ventas, 2025, 12))
print(resumir(filtrar_mes(ventas, 2025, 12), "categoria"))

                                           indicador      valor
0                                            Período    12/2025
1                                Ventas totales (S/)  794514.97
2                                   Número de ventas       2779
3                                 Venta mediana (S/)      184.0
4  Variación contra el mismo mes del año anterior...       -1.7
          categoria      total  porcentaje
0         Abarrotes  354330.70        44.6
1  Cuidado personal  146992.45        18.5
2          Limpieza  121338.25        15.3
3           Bebidas  100223.77        12.6
4            Snacks   71629.80         9.0


### Paso 3: el reporte completo

`crear_reporte` junta todas las piezas:

1. **Revisa** que haya ventas. Si escribes mal el mes, te avisa en lugar de crear un reporte vacío.
2. **Dibuja** el gráfico y lo guarda. `x` e `y` indican qué columnas usar. `plt.close()` cierra el gráfico sin mostrarlo aquí.
3. **Escribe** el Excel. `pd.ExcelWriter` abre el libro, y `with` lo guarda y lo cierra al terminar el bloque. Cada `.to_excel()` es una hoja, e `index=False` evita una columna con los números de fila.
4. Con `add_image` pega el gráfico, y con `.width` ensancha las columnas, como cuando las arrastras en Excel.

En el nombre del archivo, `{mes:02d}` escribe el mes con dos dígitos (`03`), para que los archivos queden en orden. No tienes que memorizar `ExcelWriter`, `add_image` ni `.width`: son recetas que se copian.

In [6]:
def crear_reporte(ventas, anio, mes):
    del_mes = filtrar_mes(ventas, anio, mes)

    if len(del_mes) == 0:
        print(f"No hay ventas en {mes}/{anio}. Revisa el año y el mes.")
        return

    por_ciudad = resumir(del_mes, "ciudad")

    grafico = f"grafico_{anio}_{mes:02d}.png"
    por_ciudad.plot(
        kind="bar",
        x="ciudad",
        y="total",
        legend=False,
        title=f"Ventas por ciudad, {mes}/{anio}",
        xlabel="Ciudad",
        ylabel="Soles",
        rot=0,
    )
    plt.savefig(grafico, dpi=100, bbox_inches="tight")
    plt.close()

    nombre = f"reporte_{anio}_{mes:02d}.xlsx"
    with pd.ExcelWriter(nombre, engine="openpyxl") as archivo:
        crear_indicadores(ventas, anio, mes).to_excel(archivo, sheet_name="Resumen", index=False)
        por_ciudad.to_excel(archivo, sheet_name="Ciudades", index=False)
        resumir(del_mes, "categoria").to_excel(archivo, sheet_name="Categorías", index=False)
        resumir(del_mes, "vendedor").to_excel(archivo, sheet_name="Vendedores", index=False)
        resumir(del_mes, "producto").head(10).to_excel(
            archivo, sheet_name="Top 10 productos", index=False
        )

        archivo.sheets["Resumen"].add_image(Image(grafico), "D2")

        for hoja in archivo.sheets:
            archivo.sheets[hoja].column_dimensions["A"].width = 48
            archivo.sheets[hoja].column_dimensions["B"].width = 14

    print(f"Listo: {nombre}")

### Paso 4: tu reporte, con una línea

Aquí cambias el año y el mes, y nada más. Van como números, sin comillas:

In [7]:
crear_reporte(ventas, 2025, 12)

Listo: reporte_2025_12.xlsx


El archivo aparece en la carpeta 📁 de Colab. Haz clic derecho → **Descargar** y ábrelo en Excel. Esa carpeta es temporal: descárgalo antes de cerrar.

**Nunca envíes un reporte sin comprobarlo.** Volvemos a abrir una hoja del archivo, y su suma debe coincidir con la de los datos:

In [8]:
hoja = pd.read_excel("reporte_2025_12.xlsx", sheet_name="Ciudades")

print(round(hoja["total"].sum(), 2))
print(round(filtrar_mes(ventas, 2025, 12)["total"].sum(), 2))

794514.97
794514.97


### El momento de la verdad

¿Y si te piden los 12 meses del año? `%%time`, en la primera línea de la celda, mide cuánto demora:

In [9]:
%%time
for mes in range(1, 13):
    crear_reporte(ventas, 2025, mes)

Listo: reporte_2025_01.xlsx


Listo: reporte_2025_02.xlsx


Listo: reporte_2025_03.xlsx


Listo: reporte_2025_04.xlsx


Listo: reporte_2025_05.xlsx


...

Listo: reporte_2025_08.xlsx


Listo: reporte_2025_09.xlsx


Listo: reporte_2025_10.xlsx


Listo: reporte_2025_11.xlsx


Listo: reporte_2025_12.xlsx
CPU times: user 3.69 s, sys: 697 ms, total: 4.39 s
Wall time: 4.48 s


Doce reportes en segundos, y sin errores de copiar y pegar. En la carpeta 📁 quedan los Excel y las imágenes de sus gráficos; solo necesitas los Excel. El formato de moneda y los títulos bonitos son tu siguiente paso: la IA te puede ayudar.

> 💡 **Para usarlo con tus archivos:** cambia la ruta y adapta `limpiar_ventas` a tus columnas. Si el sistema cambia el formato de la exportación, el código puede fallar o dar números errados. Por eso siempre comprobamos los totales.

## 🤖 Pídeselo a la IA

Ya sabes lo suficiente para pedir mejoras y entenderlas. Con nombres inventados:

> *"Tengo una función de Python que crea un Excel con `pd.ExcelWriter`. ¿Cómo le doy formato de moneda a la columna `total` de la hoja `Ciudades` y pongo en negrita los títulos? Explícame cada línea."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Limpiar a mano cada mes | `limpiar_ventas(archivo)` |
| Tabla dinámica | `resumir(tabla, "ciudad")` |
| Guardar un libro con varias hojas | `with pd.ExcelWriter("libro.xlsx", engine="openpyxl") as archivo:` |
| Una hoja del libro | `tabla.to_excel(archivo, sheet_name="Hoja", index=False)` |
| Pegar un gráfico en la hoja | `archivo.sheets["Hoja"].add_image(Image("grafico.png"), "D2")` |
| Rehacer el reporte cada mes | `crear_reporte(ventas, 2025, 12)` |

## 🏁 Terminaste la Temporada 0

Empezaste sin saber programar. Hoy limpias datos, cruzas tablas, haces gráficos y automatizas un reporte. En la **Temporada 1** le enseñaremos a la computadora a encontrar patrones: Machine Learning.

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/08-mini-proyecto-reporte-mensual/ejercicio.ipynb) y crea tu propio reporte por sede. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-0/08-mini-proyecto-reporte-mensual/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0